In [ ]:
# from concurrent.futures import ThreadPoolExecutor, as_completed
# from rlm_sec.filings import sec_data
# from rlm_sec.filings.utils import company_to_ticker
# import asyncio

# from rlm_sec.trainer import hf_dataloader

# all_tickers_years = [(i,j) for i,j in zip(list(combined_qa['ticker_or_company_name']),list(combined_qa['year']))]
# all_tickers_years = list(set(all_tickers_years))
# combined_qa = hf_dataloader.load_combined_qa()

# # Function to call sec_main for a given (ticker, year)
# def fetch_sec_main(args):
#     ticker, year = args
#     ticker = company_to_ticker(ticker)
#     # sec_main is async, so run it with asyncio
#     if not ticker:
#         return None, None, None
#     return (ticker, year, asyncio.run(sec_data.sec_main(ticker, year)))

# results = []
# with ThreadPoolExecutor() as executor:
#     # Submit all (ticker, year) pairs for execution
#     futures = [executor.submit(fetch_sec_main, (ticker, year)) for ticker, year in all_tickers_years]
#     for future in as_completed(futures):
#         try:
#             ticker, year, value = future.result()
#             if not ticker:
#                 continue
#             results.append((ticker, year, value))
#         except Exception as e:
#             print(f"Error fetching ({ticker}, {year}): {e}")


In [ ]:
# import re
# from pathlib import Path
# from settings import env_settings
# from rlm_sec.dataloader.vector_store import FaissVectorIndex

# # Root containing one directory per "TICKER-YYYY" with *.md inside each.
# MARKDOWN_ROOT = Path("localworkspace/markdown/sec_data/")
# FORCE_REBUILD = False

# # Directory names must end with -YYYY (handles tickers like BRK-B-2025).
# _TICKER_YEAR_DIR = re.compile(r"^(?P<ticker>.+)-(?P<year>\d{4})$")

# index = FaissVectorIndex()
# all_keys = []

# for sub in sorted(MARKDOWN_ROOT.iterdir()):
#     if not sub.is_dir():
#         continue
#     m = _TICKER_YEAR_DIR.match(sub.name)
#     if not m:
#         print(f"skip (not TICKER-YYYY): {sub.name}")
#         continue
#     ticker, year = m["ticker"], m["year"]
#     md_paths = sorted(sub.glob("*.md"))
#     if not md_paths:
#         print(f"skip (no .md): {sub.name}")
#         continue
#     try:
#         keys = index.from_markdown(
#             ticker=ticker,
#             year=year,
#             markdown_paths=md_paths,
#             force=True,
#         )
#     except Exception:
#         pass
#     all_keys.extend(keys)
#     print(f"indexed {sub.name}: {len(keys)} filing(s)")

# print(f"total index keys: {len(all_keys)}")

## TESTING ENVIRONMENT

In [1]:
from rlm_sec.trainer import hf_dataloader

combined_qa = hf_dataloader.load_combined_qa()
# all_tickers_years = [(i,j) for i,j in zip(list(combined_qa['ticker_or_company_name']),list(combined_qa['year']))]
# all_tickers_years = list(set(all_tickers_years))

/home/recoverx/astarag/recursive-lm-sec-filings/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 150/150 [00:00<00:00, 3994.12 examples/s]


In [2]:
combined_qa[-400]

{'prompt': [{'role': 'system',
   'content': 'You are a helpful and harmless assistant.'},
  {'role': 'user',
   'content': 'Answer the given question. You must conduct reasoning inside <think> and </think> first every time you get new information. After reasoning, if you lack some knowledge, you can call one of the two search tools below.\n\nTool 1 — SEC Filings (annual and quarterly reports):\n  <search>SECFilingTool(ticker, year, filing_type)</search>\n  filing_type is one of: 10-K (annual), 10-Q1, 10-Q2, 10-Q3 (quarterly).\n  Example: <search>SECFilingTool(AAPL, 2023, 10-K)</search>\n\nTool 2 — Earnings Call Transcripts:\n  <search>EarningsTranscriptTool(ticker, year, quarter)</search>\n  quarter is one of: Q1, Q2, Q3, Q4.\n  Example: <search>EarningsTranscriptTool(MSFT, 2023, Q2)</search>\n\nThe search engine will return results between <information> and </information>. You can search as many times as needed. Once you have sufficient information, provide the final answer inside <a

In [ ]:
from datasets import load_dataset

# Load the "validation.parquet" file from the local directory using HuggingFace datasets
dataset = load_dataset("parquet", data_files="data/searchR1/validation.parquet")["train"]

dataset

Generating train split: 51713 examples [00:00, 72419.97 examples/s]

Dataset({
    features: ['data_source', 'prompt', 'ability', 'env_class', 'reward_spec', 'extra_info', 'metadata'],
    num_rows: 51713
})


In [16]:
dataset[0]

{'data_source': 'searchR1_nq',
 'prompt': [{'content': 'You are a helpful and harmless assistant.',
   'role': 'system'},
  {'content': 'Answer the given question. You must conduct reasoning inside <think> and </think> first every time you get new information. After reasoning, if you find you lack some knowledge, you can call a search engine by <search> query </search> and it will return the top searched results between <information> and </information>. You can search as many times as you want. If you find no further external knowledge needed, you can directly provide the answer inside <answer> and </answer>, without detailed illustrations. For example, <answer> Beijing </answer>. Question: who got the first nobel prize in physics?',
   'role': 'user'}],
 'ability': 'fact-reasoning',
 'env_class': 'search',
 'reward_spec': {'ground_truth': {'target': ['Wilhelm Conrad Röntgen']},
  'style': 'rule'},
 'extra_info': {'index': 0,
  'need_tools_kwargs': True,
  'question': 'who got the firs

In [2]:
import requests
from settings import env_settings

url = f"{env_settings.server_url}/vector_store/search"
r = requests.post(
    url,
    json={
        "ticker": "AAPL",
        "year": "2023",
        "filing_type": "10-K",
        "query": "revenue recognition policy",
        "top_k": 3,
    },
    timeout=30,
)
r.raise_for_status()
chunks = r.json()
# for c in chunks:
#     print(c.get("score"), c.get("text", ""))
chunks

[{'text': 'Other Current Liabilities',
  'chunk_type': 'text',
  'page_num': 78,
  'section_title': 'Item 8. Financial Statements and Supplementary Data',
  'chunk_index': 150,
  'score': 0.5170146822929382},
 {'text': 'Other Non-Current Liabilities',
  'chunk_type': 'text',
  'page_num': 78,
  'section_title': 'Item 8. Financial Statements and Supplementary Data',
  'chunk_index': 152,
  'score': 0.5166018009185791},
 {'text': 'The Company has identified up to three performance obligations regularly included in arrangements involving the sale of iPhone, Mac, iPad and certain other products. The first performance obligation, which represents the substantial portion of the allocated sales price, is the hardware and bundled software delivered at the time of sale. The second performance obligation is the right to receive certain product-related bundled services, which include iCloud®, Siri® and Maps. The third performance obligation is the right to receive, on a when-and-if-available basi

In [2]:
import json

from rlm_sec.envs.finance_env import FinanceSearchEnv, SearchEnvConfig
from settings import env_settings

cfg = SearchEnvConfig(
    search_url=f"{env_settings.server_url}/vector_store/search",
    topk=3,
    timeout=30,
    log_requests=True,
)

extras = {"max_turns": 2}
env = FinanceSearchEnv(cfg, extras=extras)

# Required for reward on terminal step (not set in SECSearchEnv today):
env.ground_truth = {"target": "some canonical answer string"}

# system_user_messages, meta = env.init(
#     [
#         {"role": "user", "content": "Answer the question using search. Wrap queries in <search>...</search> and the final answer in <answer>...</answer>."},
#     ]
# )

# Simulate first model turn: search
out1 = env.step("<search>SECFilingTool(cash flow from operations, AAPL, 2023, 10-Q3)</search>")
print("done:", out1["done"], "reward:", out1["reward"])
print("metadata:", out1["metadata"])
if out1["observations"]:
    obs = out1["observations"][0]["content"]
    print("observation (prefix):", obs[:1500])

# Simulate second turn: final answer (ends episode)
out2 = env.step("<answer>some canonical answer string</answer>")
print("done:", out2["done"], "reward:", out2["reward"])

[Search Request ID: e73a39da-ab08-48a8-91ea-9c74af02be73] API Request Failed after 10 attempts: [Search Request ID: e73a39da-ab08-48a8-91ea-9c74af02be73] API Request Error: 404 Client Error: Not Found for url: http://127.0.0.1:8888/tools/sec_filing_to_markdown_embed_and_search
Batch search: API error occurred: [Search Request ID: e73a39da-ab08-48a8-91ea-9c74af02be73] API Request Error: 404 Client Error: Not Found for url: http://127.0.0.1:8888/tools/sec_filing_to_markdown_embed_and_search


done: False reward: 0
metadata: {'tool_group': 'SECFilingToolGroup', 'tool_name': 'sec_filing_to_markdown_embed_and_search', 'tool_input': ParsedSearch(query='cash flow from operations', ticker='AAPL', year='2023', filing_type_or_quarter='10-Q3', tool_group_name='SECFilingToolGroup', tool_name='sec_filing_to_markdown_embed_and_search'), 'tool_metadata': {}}
observation (prefix): 
<information>
Search error: [Search Request ID: e73a39da-ab08-48a8-91ea-9c74af02be73] API Request Error: 404 Client Error: Not Found for url: http://127.0.0.1:8888/tools/sec_filing_to_markdown_embed_and_search</information>

done: True reward: 1.0
